In [ ]:
from pathlib import Path
from tqdm.auto import tqdm

import pandas as pd
import geopandas as gpd
import numpy as np
from matplotlib import pyplot as plt
import cartopy
import cmocean
import xarray as xr

from load_tuning_results import (
    load_results_raw,
    load_result_for_key,
    add_derived_features,
    filter_suspicious_routes,
    load_results,
    get_seed_routes_gdf,
    get_forcing_paths_df,
)

import warnings

warnings.filterwarnings("ignore")

In [ ]:
# parameters

# extent
lon_min, lon_max = -80 - 5, -10 + 5
lat_min, lat_max = 25 - 5, 55 + 5

# results dataframe from
gpq_file = "results/results_prelim.geoparquet"

# single results file
results_file = "results/results_19430020.msgpack"

# Configure route visualization styles

# Color mapping for journey direction (easily customizable)
ROUTE_COLORS = {
    "Atlantic_forward": "magenta",
    "Atlantic_backward": "orange",
}

# Linestyle mapping for hazard enforcement
ROUTE_LINESTYLES = {
    False: "-",  # solid = hazards enforced
    True: "--",  # dashed = hazards ignored
}

# Route visualization parameters
ROUTE_LINEWIDTH = 1.5
ROUTE_ALPHA = 1.0
SEED_COLOR = "cyan"
SEED_LINEWIDTH = 1.5
SEED_ALPHA = 1.0

In [ ]:
# Create style lookup helper function
def get_route_style(journey_name, ignore_hazards):
    """Get plotting style for route configuration."""
    return {
        "color": ROUTE_COLORS[journey_name],
        "linestyle": ROUTE_LINESTYLES[ignore_hazards],
        "linewidth": ROUTE_LINEWIDTH,
        "alpha": ROUTE_ALPHA,
    }

In [ ]:
print(lon_min, lon_max, lat_min, lat_max)

lon_cent = (lon_min + lon_max) / 2.0
lat_cent = (lat_min + lat_max) / 2.0
print(lon_cent, lat_cent)

In [ ]:
gdf = gpd.read_parquet(gpq_file)
gdf = add_derived_features(gdf)
gdf

In [ ]:
# Load results and extract seed routes
results = load_results([results_file])
gdf_seed = get_seed_routes_gdf(results)
gdf_seed

In [ ]:
# map start times to month (to match with figure)
gdf["time_month"] = pd.to_datetime(gdf.journey_time_start.astype(str)).dt.strftime(
    "%Y-%m"
)

# select elites based on minimum abs cost.
best_elites_gdf = (
    gdf.groupby(["journey_name", "time_month", "hyper_ignore_hazards"], dropna=False)
    .apply(lambda g: g.loc[g.elite_cost_absolute.idxmin()])
    .reset_index(drop=True)
)

# convert back to gdf
best_elites_gdf = gpd.GeoDataFrame(best_elites_gdf, geometry="geometry")

In [ ]:
best_elites_gdf

In [ ]:
# Extract forcing paths
forcing_df = get_forcing_paths_df(results)
forcing_currents_path = forcing_df["forcing_currents_path"].iloc[0]
print(forcing_currents_path)
forcing_waves_path = forcing_df["forcing_waves_path"].iloc[0]
print(forcing_waves_path)
forcing_winds_path = forcing_df["forcing_winds_path"].iloc[0]
print(forcing_winds_path)

In [ ]:
ds_currents = xr.open_zarr(forcing_currents_path)
ds_currents = ds_currents.sel(
    longitude=slice(lon_min, lon_max), latitude=slice(lat_min, lat_max)
)
ds_currents = ds_currents.assign(
    speed=(ds_currents.to_array() ** 2).sum("variable") ** 0.5
)
ds_currents = ds_currents.resample(time="1M").mean().compute()
ds_currents = ds_currents.where(ds_currents.speed > 0)
ds_currents

In [ ]:
ds_waves = xr.open_zarr(forcing_waves_path)
ds_waves = ds_waves.sel(
    longitude=slice(lon_min, lon_max), latitude=slice(lat_min, lat_max)
)
ds_waves = ds_waves.resample(time="1M").quantile(0.9).compute()
ds_waves

In [ ]:
ds_winds = xr.open_zarr(forcing_winds_path)
ds_winds = ds_winds.sel(
    longitude=slice(lon_min, lon_max), latitude=slice(lat_min, lat_max)
)
ds_winds = ds_winds.assign(
    speed=(ds_winds.to_array() ** 2).sum("variable").compute() ** 0.5
)
ds_winds = ds_winds.resample(time="1M").quantile(0.9)  # .compute()
ds_winds

In [ ]:
times = sorted(list(best_elites_gdf.time_month.unique()))
times

In [ ]:
fig, ax = plt.subplots(
    len(times),
    3,
    subplot_kw={
        "projection": cartopy.crs.Stereographic(
            central_latitude=lat_cent, central_longitude=lon_cent
        )
    },
    figsize=(len(times) * 2.5, 3 * 4),
    sharex=True,
    sharey=True,
)

for n in range(len(times)):
    _time = times[n]
    # Filter elite routes for this time period
    time_elites = best_elites_gdf[best_elites_gdf.time_month == _time]

    for m in range(3):
        _ax = ax[n, m]

        # Plot seed route (reference route)
        gdf_seed.iloc[:1].plot(
            ax=_ax,
            transform=cartopy.crs.PlateCarree(),
            color=SEED_COLOR,
            linewidth=SEED_LINEWIDTH,
            alpha=SEED_ALPHA,
            label="Seed route",
        )

        # Plot elite routes for all configurations
        for journey_name in ["Atlantic_forward", "Atlantic_backward"]:
            for ignore_hazards in [False, True]:
                # Filter for this specific configuration
                config_routes = time_elites[
                    (time_elites.journey_name == journey_name)
                    & (time_elites.hyper_ignore_hazards == ignore_hazards)
                ]

                # Plot if routes exist for this configuration
                if len(config_routes) > 0:
                    hazard_label = "no hazards" if ignore_hazards else "with hazards"
                    direction_label = journey_name.split("_")[1]
                    style = get_route_style(journey_name, ignore_hazards)
                    config_routes.plot(
                        ax=_ax,
                        transform=cartopy.crs.PlateCarree(),
                        label=f"{direction_label} ({hazard_label})",
                        **style,
                    )

        # Plot forcing data
        if m == 0:
            ds_currents.speed.sel(time=_time).plot(
                ax=_ax,
                transform=cartopy.crs.PlateCarree(),
                vmin=0,
                vmax=1.5,
                extend="max",
                cmap=cmocean.cm.speed,
                cbar_kwargs={"label": "speed (m/s)"},
                rasterized=True,
            )
            _title = f"mean current speed: {_time}"
        if m == 1:
            ds_waves.VHM0.sel(time=_time).plot(
                ax=_ax,
                transform=cartopy.crs.PlateCarree(),
                vmin=0,
                vmax=6.0,
                extend="max",
                cmap=cmocean.cm.amp_i_r,
                cbar_kwargs={"label": "wave height (m)"},
                rasterized=True,
            )
            _title = f"q90% wave height: {_time}"
        if m == 2:
            ds_winds.speed.sel(time=_time).plot(
                ax=_ax,
                transform=cartopy.crs.PlateCarree(),
                vmin=0,
                vmax=15.0,
                extend="max",
                cmap=cmocean.cm.speed_i_r,
                cbar_kwargs={"label": "wind speed (m/s)"},
                rasterized=True,
            )
            _title = f"q90% wind speed: {_time}"

        _ax.coastlines()
        _ax.gridlines()
        _ax.set_title(_title)
        _ax.set_extent([lon_min, lon_max, lat_min, lat_max])

fig.tight_layout()

# Adjust bottom margin to make space for legend (reduced from 0.08 to 0.04)
fig.subplots_adjust(bottom=0.04)

# Add legend to avoid clutter, remove duplicates
handles, labels = ax[0, 0].get_legend_handles_labels()
by_label = dict(zip(labels, handles))
fig.legend(
    by_label.values(),
    by_label.keys(),
    loc="upper center",
    bbox_to_anchor=(0.5, -0.01),
    ncol=6,
    frameon=False,
    fontsize=9,
)

# Save with bbox_inches='tight' to include legend
fig.savefig(
    "../figures/test_cases_journey_and_forcing_overview_with_elites.png",
    dpi=200,
    bbox_inches="tight",
)
fig.savefig(
    "../figures/test_cases_journey_and_forcing_overview_with_elites.pdf",
    dpi=200,
    bbox_inches="tight",
)

In [ ]:
import seaborn as sns

In [ ]:
best_elites_gdf = best_elites_gdf.assign(
    journey_name_short=best_elites_gdf["journey_name"].apply(lambda s: s.split("_")[-1])
)
best_elites_gdf = best_elites_gdf.assign(
    hazards_included=~best_elites_gdf.hyper_ignore_hazards
)
best_elites_gdf.hazards_included

In [ ]:
fig, ax = plt.subplots(1, 2)

sns.heatmap(
    (
        -100
        + 100
        * best_elites_gdf.groupby(
            ["journey_name_short", "time_month", "hazards_included"]
        )
        .elite_cost_relative.first()
        .loc["forward"]
        .unstack(-1)
    ).astype(float),
    annot=True,
    fmt=".1f",
    ax=ax[0],
    # colorbar=False
)
for t in ax[0].texts:
    t.set_text(t.get_text() + "%")

sns.heatmap(
    (
        -100
        + 100
        * best_elites_gdf.groupby(
            ["journey_name_short", "time_month", "hazards_included"]
        )
        .elite_cost_relative.first()
        .loc["backward"]
        .unstack(-1)
    ).astype(float),
    annot=True,
    fmt=".1f",
    ax=ax[1],
    # colorbar=False
)
for t in ax[1].texts:
    t.set_text(t.get_text() + "%")

ax[0].set_title("forward")
ax[1].set_title("backward")

fig.tight_layout()

fig.savefig(
    "../figures/test_cases_best_cost_reduction_per_overview_case.png",
    dpi=200,
    bbox_inches="tight",
)
fig.savefig(
    "../figures/test_cases_best_cost_reduction_per_overview_case.pdf",
    dpi=200,
    bbox_inches="tight",
)

In [ ]:
# Get both forward and backward
both_tables = (
    (
        -100
        + 100
        * best_elites_gdf.groupby(
            ["journey_name_short", "time_month", "hazards_included"]
        )
        .elite_cost_relative.first()
        .unstack(0)
        .unstack(-1)
    )
    .astype(float)
    .round(1)
)

# Flatten multi-level columns: (direction, hazards_included)
both_tables.columns = [
    f"{direction}_{hazard}" for direction, hazard in both_tables.columns
]

print(both_tables.to_latex())

In [ ]:
# Get both forward and backward
both_tables = (
    (
        best_elites_gdf.groupby(
            ["journey_name_short", "time_month", "hazards_included"]
        )
        .elite_cost_absolute.first()
        .unstack(0)
        .unstack(-1)
    )
    .astype(float)
    .round(2)
)

# Flatten multi-level columns: (direction, hazards_included)
both_tables.columns = [
    f"{direction}_{hazard}" for direction, hazard in both_tables.columns
]

print(both_tables.to_latex())

In [ ]:
# Get both forward and backward
both_tables = (
    (
        best_elites_gdf.groupby(
            ["journey_name_short", "time_month", "hazards_included"]
        )
        .elite_cost_absolute.first()
        .unstack(0)
        .unstack(-1)
        / 1e12
    )
    .astype(float)
    .round(2)
)

# Flatten multi-level columns: (direction, hazards_included)
both_tables.columns = [
    f"{direction}_{hazard}" for direction, hazard in both_tables.columns
]

print(both_tables.to_latex())

In [ ]:
(0.2 * best_elites_gdf.elite_cost_absolute / 42e9 * 3.15).plot.hist()